## Descion Tree From Scratch (Binary Result)

In [31]:
# Import Packages
import pandas as pd
import math
from pprint import pprint # clear dictionay printing

In [32]:
# DataSet Example
dataset = pd.read_csv("./play_tennis_dataSet.csv")
dataset = dataset.drop("day", axis=1)

# Based on Dataset Make Data Structure
newDataSet = {}
for col in dataset:
    colValue = dataset[col]
    newDataSet[col] = colValue.tolist()
print(newDataSet)

{'outlook': ['sunny', 'sunny', 'overcast', 'rain', 'rain', 'overcast', 'sunny', 'sunny', 'rain', 'rain', 'sunny', 'overcast', 'overcast', 'rain'], 'temp': ['hot', 'hot', 'hot', 'mild', 'cool', 'cool', 'mild', 'cool', 'mild', 'cool', 'mild', 'mild', 'cool', 'mild'], 'play': [0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0]}


## Get Root Element By Calculate Entropy For Eash Feature

In [33]:
goalFeature = "play"
dataSetLen = len(newDataSet[goalFeature])

def getFeatureValue(feature):
    featureValue = newDataSet[feature]
    newFeatureValue = []
    for i in range(0, len(featureValue)):
        if featureValue[i] not in newFeatureValue: newFeatureValue.append(featureValue[i])
    return newFeatureValue
# getFeatureValue("play")


# Get S for Main DataSet
def getSOfDataSet(dataset):
    S = [dataset[goalFeature].count(0), dataset[goalFeature].count(1)]
    return S


# Get S For Specific Feature
def getSForSpecificFeatureValue(feature, dataset, featureValue):
    colOfFeauture = dataset[feature]
    colOfGoal = dataset[goalFeature]
    # print(colOfFeauture)
    # print(colOfGoal)

    S = []
    countOfNeg = 0
    countOfPos = 0
    for i in range(0, dataSetLen):
        if (colOfFeauture[i] == featureValue):
            if (colOfGoal[i] == 0): countOfNeg += 1
            if (colOfGoal[i] == 1): countOfPos += 1
    
    S = [countOfPos, countOfNeg]
    # print(S)
    return S

# getSForSpecificFeatureValue("outlook", newDataSet, "sunny")

def getEntropyOfS(S):
    sumOfS = sum(S)
    if S[0] == S[1]:
        return 1
    if sumOfS == 0 or S[0] == 0 or S[1] == 0:
        return 0
    p0 = S[0] / sumOfS
    p1 = S[1] / sumOfS
    e = - p0 * math.log2(p0) - p1 * math.log2(p1)
    return round(e, 2)


# Calculate Entropy
def calculateGain(dataset):
    gainOfFeature = []
    for feature in dataset:
        if (feature != goalFeature):
            # print("----- Feature:", feature)
            featureValues = getFeatureValue(feature=feature)
            # print(f"featureValues = {featureValues}")
            entropyOfFeatures = []
            for featureValue in featureValues:
                S = getSForSpecificFeatureValue(feature=feature, dataset=dataset, featureValue=featureValue)

                # Caculate Entropy
                e = getEntropyOfS(S)
                entropyOfFeatures.append({featureValue: e, "S": S})
                # print(f"feature: {featureValue} and Entropy = {round(e, 2)} and S = {S}")

            # Calculate Gain Of Feature
            # print(entropyOfFeatures)
            S_total = getSOfDataSet(dataset)
            gain = getEntropyOfS(S_total)
            for entropyFeature in entropyOfFeatures:
                featureValue = list(entropyFeature.keys())[0]
                S_v = entropyFeature['S']
                weight = sum(S_v) / sum(S_total)
                gain -= entropyFeature[featureValue] * weight

            gainOfFeature.append({feature: gain})
    return gainOfFeature

gainOfFeature = calculateGain(newDataSet)
print(gainOfFeature)


# Get Root Element
def getRootElement(gainOfFeature):
    max_val = list(gainOfFeature[0].values())[0]
    bestFeature = list(gainOfFeature[0].keys())[0]
    print(max_val)
    for feature in gainOfFeature:
        val = list(feature.values())[0]
        if (max_val <= val):
            max_val = val
            bestFeature = list(feature.keys())[0]
    return bestFeature

bestFeature = getRootElement(gainOfFeature=gainOfFeature)
print(f"best feature is {bestFeature}")

[{'outlook': 0.2471428571428571}, {'temp': 0.09142857142857136}]
0.2471428571428571
best feature is outlook


## Build a Descion Tree

In [48]:
def create_node(featureName, featureValue, entropy, S):
    return {
        'featureName': featureName,
        'featureValue': featureValue,
        'entropy': entropy,
        'S': S,
        'children': []
    }

def add_child(node, obj):
    node['children'].append(obj)

def displayTree(rootElement):
    print("--> ", rootElement['featureName'], rootElement['featureValue'],
        rootElement['entropy'], rootElement['S'])
    for node in rootElement['children']:
        displayTree(node)

# Intialize The Tree And return first Node in Tree
def initTree():
    # Get Best Feature
    gainOfFeature = calculateGain(dataset=newDataSet)
    bestFeature = getRootElement(gainOfFeature=gainOfFeature)

    # Calculate S and entropy total
    S_total = getSOfDataSet(dataset=newDataSet)
    entropy_total = getEntropyOfS(S=S_total)
    
    # Create Root 
    root = create_node(featureName=bestFeature, featureValue="ROOT", entropy=entropy_total, S=S_total)

    # Get Feature Value
    featureValues = getFeatureValue(feature=bestFeature)

    # Get Entropy For Each Feature
    for value in featureValues:
        S_v = getSForSpecificFeatureValue(feature=bestFeature, dataset=newDataSet, featureValue=value)
        entropy_Sv = getEntropyOfS(S=S_v)
        
        child = create_node(featureName=bestFeature, featureValue=value, entropy=entropy_Sv, S=S_v)
        add_child(root, child)

    return root

root = initTree()
# pprint(root)

def getNewDatasetBasedOnFeatureValue(dataset, featureVal):
    pprint(dataset)

# Build a Tree
def buidTree(dataset, root):
    for child in root["children"]:
        # Extract New Dataset Based on Feature
        extractedDataSet = getNewDatasetBasedOnFeatureValue(dataset=dataset, featureVal=child['featureValue'])
        print(extractedDataSet)
        
        # Get Entropy, S,

        # DataSet Should Be Updated

        # Recursive Function
        if (len(child["children"])):
            print("Should Call BuitTree")
            pass
        
        print(child)
buidTree(dataset=newDataSet, root=root)

0.2471428571428571
{'outlook': ['sunny',
             'sunny',
             'overcast',
             'rain',
             'rain',
             'overcast',
             'sunny',
             'sunny',
             'rain',
             'rain',
             'sunny',
             'overcast',
             'overcast',
             'rain'],
 'play': [0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0],
 'temp': ['hot',
          'hot',
          'hot',
          'mild',
          'cool',
          'cool',
          'mild',
          'cool',
          'mild',
          'cool',
          'mild',
          'mild',
          'cool',
          'mild']}
None
{'featureName': 'outlook', 'featureValue': 'sunny', 'entropy': 0.97, 'S': [2, 3], 'children': []}
{'outlook': ['sunny',
             'sunny',
             'overcast',
             'rain',
             'rain',
             'overcast',
             'sunny',
             'sunny',
             'rain',
             'rain',
             'sunny',
             'o